# 🔬 Naive Bayes vs. Logistic Regression for Bio-Image Classification

**Subtitle:** A probabilistic classifier comparison on HeLa fluorescence microscopy images

| | |
|---|---|
| **Hardware Target** | CPU only (no GPU required) |
| **Platform** | Google Colab (free tier) |
| **Core Frameworks** | scikit-learn, scikit-image, NumPy, Matplotlib, Seaborn |
| **Prerequisites** | Module 04 — Geometry-Based Classification |
| **Estimated Runtime** | ~5 minutes |
| **Contributor** | Rishita Prabhakar Polumatla |

> ⚠️ **Disclaimer:** This tutorial is for **research and educational purposes only**. Not intended for clinical diagnosis.

---

## 🎯 What You Will Learn

1. How **Gaussian Naive Bayes** works — Bayes' theorem + independence assumption
2. How **Logistic Regression** works — linear decision boundary + softmax
3. **When each classifier wins** — small data, correlated features, calibration
4. How to **compare classifiers fairly** — same features, cross-validation
5. How to **interpret model outputs** — confusion matrices, calibration curves, feature weights
6. How these relate to SVM/k-NN from Module 04

---

## 📋 Table of Contents

1. Environment Setup & Hardware Check
2. Theory: Naive Bayes
3. Theory: Logistic Regression
4. Dataset Loading (HeLa)
5. Feature Extraction (GLCM + Geometric + Haralick)
6. Preprocessing & Normalisation
7. Model Training: Gaussian Naive Bayes
8. Model Training: Logistic Regression
9. Evaluation & Side-by-Side Comparison
10. Calibration Curves
11. Feature Weight Visualisation
12. Regularisation Sweep
13. Discussion Questions & Bonus Exercises

## 1. Environment Setup & Hardware Check

In [ ]:
# ── Hardware Check ──
import subprocess, sys

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout if result.returncode == 0 else 'No GPU — CPU mode (fine for this tutorial ✅)')
except FileNotFoundError:
    print('No GPU detected — CPU mode ✅')

print(f'Python: {sys.version}')

In [ ]:
# ── Install Dependencies ──
!pip install -q scikit-learn scikit-image numpy matplotlib seaborn mahotas

In [ ]:
# ── Global Imports & Configuration ──
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from skimage import io, color, transform, filters
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import regionprops, label

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline

import mahotas

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.dpi': 100,
    'figure.figsize': (10, 6),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

CLASS_NAMES = [
    'Endosome', 'Lysosome', 'Mitochondria', 'Nucleus',
    'Endoplasmic Reticulum', 'Actin', 'Golgi', 'Plasma Membrane',
    'Microtubule', 'Nucleolus'
]

print('✅ All imports successful.')

## 2. Theory: Naive Bayes

### 2.1 Bayes' Theorem

Bayes' theorem computes the probability of class $y$ given observed features $\mathbf{x}$:

$$P(y \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid y)\, P(y)}{P(\mathbf{x})}$$

- $P(y)$ — **prior**: how common is class $y$ in training data?
- $P(\mathbf{x} \mid y)$ — **likelihood**: how probable are these features given class $y$?
- $P(y \mid \mathbf{x})$ — **posterior**: what we actually want

Since $P(\mathbf{x})$ is constant across classes, the decision rule is:

$$\hat{y} = \arg\max_y\, P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

### 2.2 The "Naive" Independence Assumption

Naive Bayes assumes all features are **conditionally independent** given the class:

$$P(\mathbf{x} \mid y) = \prod_{i=1}^{n} P(x_i \mid y)$$

This is rarely true for image features — GLCM contrast and homogeneity are correlated. Yet it still works surprisingly well in practice.

### 2.3 Gaussian Naive Bayes

For continuous features, each $P(x_i \mid y)$ is modelled as a Gaussian:

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_{y,i}^2}} \exp\!\left(-\frac{(x_i - \mu_{y,i})^2}{2\sigma_{y,i}^2}\right)$$

**Training:** just compute $\mu_{y,i}$ and $\sigma_{y,i}^2$ per class and feature — no iterative optimisation, extremely fast.

> **Key insight:** Naive Bayes is *generative* — it models $P(x \mid y)$ then inverts via Bayes' theorem.

## 3. Theory: Logistic Regression

### 3.1 Softmax for Multi-Class

Logistic Regression is *discriminative* — it models $P(y \mid \mathbf{x})$ directly via the **softmax**:

$$P(y=k \mid \mathbf{x}) = \frac{e^{\mathbf{w}_k \cdot \mathbf{x} + b_k}}{\sum_{j=1}^{K} e^{\mathbf{w}_j \cdot \mathbf{x} + b_j}}$$

### 3.2 Training: Cross-Entropy + L2 Regularisation

$$\mathcal{L}_{\text{reg}} = -\frac{1}{N}\sum_{i=1}^{N} \log P(y_i \mid \mathbf{x}_i) + \frac{\lambda}{2} \|\mathbf{w}\|^2$$

Controlled by $C = 1/\lambda$ (default $C=1$):
- **Smaller $C$** → stronger regularisation → simpler model
- **Larger $C$** → weaker regularisation → fits training data more closely

### 3.3 Generative vs. Discriminative

| | Naive Bayes (Generative) | Logistic Regression (Discriminative) |
|---|---|---|
| **Models** | $P(x \mid y)$ then inverts | $P(y \mid x)$ directly |
| **Training** | Closed-form | Iterative gradient descent |
| **Small data** | ✅ Excellent | ⚠️ May overfit |
| **Correlated features** | ❌ Violated assumption | ✅ Handled naturally |
| **Probability calibration** | ❌ Often overconfident | ✅ Well calibrated |

## 4. Dataset Loading (HeLa Cell Organelle Dataset)

In [ ]:
# ── Load or Synthesise HeLa Dataset ──
DATA_DIR = './hela_data'

def load_hela_dataset(data_dir, img_size=(64, 64), max_per_class=50):
    """Load HeLa dataset from disk. Returns (images, labels, class_names)."""
    images, labels, class_names = [], [], []
    if not os.path.exists(data_dir):
        return None, None, None
    class_dirs = sorted([d for d in os.listdir(data_dir)
                         if os.path.isdir(os.path.join(data_dir, d))])
    for cls_idx, cls_name in enumerate(class_dirs):
        cls_path = os.path.join(data_dir, cls_name)
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg'))][:max_per_class]
        for fname in files:
            try:
                img = io.imread(os.path.join(cls_path, fname))
                if img.ndim == 3:
                    img = color.rgb2gray(img)
                img = transform.resize(img, img_size, anti_aliasing=True)
                images.append(img.astype(np.float64))
                labels.append(cls_idx)
            except Exception:
                pass
        class_names.append(cls_name)
    return (np.array(images), np.array(labels), class_names) if images else (None, None, None)


def generate_synthetic_dataset(n_classes=10, n_per_class=60, img_size=(64, 64)):
    """Generate synthetic bio-image-like data as a fallback."""
    rng = np.random.RandomState(SEED)
    images, labels = [], []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            base = np.zeros(img_size)
            for _ in range(cls + 2):
                cx, cy = rng.randint(10, img_size[0]-10), rng.randint(10, img_size[1]-10)
                radius = rng.randint(3, 10 + cls)
                yy, xx = np.ogrid[:img_size[0], :img_size[1]]
                base[(xx - cx)**2 + (yy - cy)**2 <= radius**2] = 1.0
            noise = rng.normal(0, 0.05 + cls * 0.02, img_size)
            img = filters.gaussian(np.clip(base + noise, 0, 1), sigma=1.0 + cls * 0.3)
            images.append(img)
            labels.append(cls)
    return np.array(images), np.array(labels), CLASS_NAMES[:n_classes]


images, labels, class_names = load_hela_dataset(DATA_DIR)
if images is None:
    print('ℹ️  HeLa data not found — using synthetic bio-image dataset.')
    images, labels, class_names = generate_synthetic_dataset()
else:
    print('✅ Loaded real HeLa dataset.')

print(f'Dataset: {images.shape}, Classes: {len(class_names)}, Samples/class: {np.bincount(labels)}')

In [ ]:
# ── Visualise Sample Images ──
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images — One Per Class', fontsize=14, fontweight='bold')
for ax, cls in zip(axes.flat, range(min(10, len(class_names)))):
    idx = np.where(labels == cls)[0][0]
    ax.imshow(images[idx], cmap='gray')
    ax.set_title(class_names[cls], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Feature Extraction (GLCM + Geometric + Haralick)

Same feature pipeline as Module 04 — the only thing changing is the classifier.

In [ ]:
# ── Feature Extraction Functions ──

def extract_glcm_features(img):
    """Extract 5 GLCM texture features (contrast, dissimilarity, homogeneity, energy, correlation)."""
    img_q = (img * 15).astype(np.uint8)
    glcm = graycomatrix(img_q, distances=[1],
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=16, symmetric=True, normed=True)
    return np.array([graycoprops(glcm, p).mean()
                     for p in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']])


def extract_geometric_features(img):
    """Extract 6 geometric/morphological features using skimage regionprops."""
    binary = img > filters.threshold_otsu(img)
    props = regionprops(label(binary))
    if not props:
        return np.zeros(6)
    region = max(props, key=lambda r: r.area)
    return np.array([region.area, region.perimeter, region.eccentricity,
                     region.solidity, region.extent, region.equivalent_diameter_area])


def extract_haralick_features(img):
    """Extract 13 Haralick texture features using Mahotas (averaged over directions)."""
    img_q = (img * 255).astype(np.uint8)
    if img_q.sum() == 0:
        return np.zeros(13)
    try:
        return mahotas.features.haralick(img_q).mean(axis=0)
    except Exception:
        return np.zeros(13)


def extract_all_features(img):
    """Combine GLCM (5) + Geometric (6) + Haralick (13) = 24 features."""
    return np.concatenate([
        extract_glcm_features(img),
        extract_geometric_features(img),
        extract_haralick_features(img)
    ])


# Quick test
test_feat = extract_all_features(images[0])
print(f'Feature vector length: {len(test_feat)} (5 GLCM + 6 Geometric + 13 Haralick)')

In [ ]:
# ── Extract Features for All Images ──
print('Extracting features... (~1 minute)')
feature_matrix = np.array([extract_all_features(img) for img in images])
print(f'Feature matrix: {feature_matrix.shape}  ({feature_matrix.shape[0]} images × {feature_matrix.shape[1]} features)')

## 6. Preprocessing & Normalisation

In [ ]:
# ── Handle NaN/Inf & Train/Test Split ──
n_bad = (~np.isfinite(feature_matrix)).sum()
if n_bad > 0:
    print(f'⚠️  {n_bad} NaN/Inf values found — replacing with column means.')
    col_means = np.nanmean(feature_matrix, axis=0)
    mask = ~np.isfinite(feature_matrix)
    feature_matrix[mask] = np.take(col_means, np.where(mask)[1])
else:
    print('✅ No NaN/Inf values.')

X_train, X_test, y_train, y_test = train_test_split(
    feature_matrix, labels, test_size=0.25, random_state=SEED, stratify=labels)

# StandardScaler: critical for Logistic Regression (GLCM contrast ~ [0,10000], eccentricity ~ [0,1])
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit ONLY on train data to avoid leakage
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train_s.shape[0]} samples | Test: {X_test_s.shape[0]} samples')
print(f'After scaling — mean: {X_train_s.mean():.4f} (≈0), std: {X_train_s.std():.4f} (≈1)')

In [ ]:
# ── Feature Distribution by Class ──
feature_names = (
    ['GLCM Contrast', 'GLCM Dissimilarity', 'GLCM Homogeneity', 'GLCM Energy', 'GLCM Correlation']
    + ['Area', 'Perimeter', 'Eccentricity', 'Solidity', 'Extent', 'Eq. Diameter']
    + [f'Haralick {i+1}' for i in range(13)]
)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('GLCM Feature Distributions by Class (before scaling)', fontsize=12)
for i, ax in enumerate(axes):
    for cls in range(len(class_names)):
        ax.hist(feature_matrix[labels == cls, i], bins=15, alpha=0.5, label=class_names[cls])
    ax.set_title(feature_names[i], fontsize=9)
    ax.set_xlabel('Value')
    if i == 0: ax.set_ylabel('Count')
axes[-1].legend(loc='upper right', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()
print('Overlapping distributions = harder task. Use this to assess feature discriminability.')

## 7. Model Training: Gaussian Naive Bayes

In [ ]:
# ── Train Gaussian Naive Bayes ──
gnb = GaussianNB()
gnb.fit(X_train_s, y_train)
y_pred_gnb = gnb.predict(X_test_s)
acc_gnb = accuracy_score(y_test, y_pred_gnb)

print(f'Gaussian Naive Bayes Test Accuracy: {acc_gnb:.4f} ({acc_gnb*100:.1f}%)')
print()
print(classification_report(y_test, y_pred_gnb, target_names=class_names))

In [ ]:
# ── 5-Fold Cross-Validation for Naive Bayes ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

gnb_pipe = Pipeline([('scaler', StandardScaler()), ('model', GaussianNB())])
cv_gnb = cross_val_score(gnb_pipe, feature_matrix, labels, cv=cv, scoring='accuracy')

print(f'GNB 5-Fold CV: {cv_gnb.mean():.4f} ± {cv_gnb.std():.4f}')
print(f'Per-fold: {np.round(cv_gnb, 4)}')

## 8. Model Training: Logistic Regression

In [ ]:
# ── Train Logistic Regression ──
lr = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    C=1.0,
    max_iter=1000,
    random_state=SEED
)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f'Logistic Regression Test Accuracy: {acc_lr:.4f} ({acc_lr*100:.1f}%)')
print()
print(classification_report(y_test, y_pred_lr, target_names=class_names))

In [ ]:
# ── 5-Fold Cross-Validation for Logistic Regression ──
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(multi_class='multinomial', solver='lbfgs',
                                  C=1.0, max_iter=1000, random_state=SEED))
])
cv_lr = cross_val_score(lr_pipe, feature_matrix, labels, cv=cv, scoring='accuracy')

print(f'LR 5-Fold CV : {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')
print(f'Per-fold: {np.round(cv_lr, 4)}')

## 9. Evaluation & Side-by-Side Comparison

In [ ]:
# ── Summary Table ──
print('=' * 52)
print('       CLASSIFIER COMPARISON SUMMARY')
print('=' * 52)
print(f'{"Metric":<28} {"Naive Bayes":>10} {"Logistic Reg":>12}')
print('-' * 52)
print(f'{"Test Accuracy":<28} {acc_gnb:>10.4f} {acc_lr:>12.4f}')
print(f'{"CV Mean":<28} {cv_gnb.mean():>10.4f} {cv_lr.mean():>12.4f}')
print(f'{"CV Std":<28} {cv_gnb.std():>10.4f} {cv_lr.std():>12.4f}')
print('=' * 52)
winner = 'Logistic Regression' if acc_lr > acc_gnb else ('Naive Bayes' if acc_gnb > acc_lr else 'Tie')
print(f'\n🏆 Higher test accuracy: {winner}')

In [ ]:
# ── Confusion Matrices ──
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, y_pred, title in zip(axes,
                              [y_pred_gnb, y_pred_lr],
                              ['Gaussian Naive Bayes', 'Logistic Regression']):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, y_pred):.3f}', fontsize=11)
plt.suptitle('Confusion Matrices', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-Class Accuracy ──
gnb_pc = [accuracy_score(y_test[y_test==c], y_pred_gnb[y_test==c]) if (y_test==c).sum() > 0 else 0
          for c in range(len(class_names))]
lr_pc  = [accuracy_score(y_test[y_test==c], y_pred_lr[y_test==c])  if (y_test==c).sum() > 0 else 0
          for c in range(len(class_names))]

x, w = np.arange(len(class_names)), 0.35
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, gnb_pc, w, label='Naive Bayes', color='steelblue', alpha=0.8)
ax.bar(x + w/2, lr_pc,  w, label='Logistic Regression', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.set_ylabel('Per-Class Accuracy')
ax.set_title('Per-Class Accuracy: Naive Bayes vs. Logistic Regression')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

print('NB wins on:', [class_names[i] for i in range(len(class_names)) if gnb_pc[i] > lr_pc[i]])
print('LR wins on:', [class_names[i] for i in range(len(class_names)) if lr_pc[i] > gnb_pc[i]])

## 10. Calibration Curves

A **well-calibrated** model: when it says 80% confident → it should be right ~80% of the time.

Naive Bayes is famously **overconfident** (because the independence assumption breaks down).
Logistic Regression is generally better calibrated.

In [ ]:
# ── Calibration Curve (class 0 vs rest) ──
y_binary = (y_test == 0).astype(int)
prob_gnb = gnb.predict_proba(X_test_s)[:, 0]
prob_lr  = lr.predict_proba(X_test_s)[:, 0]

frac_gnb, mean_gnb = calibration_curve(y_binary, prob_gnb, n_bins=10, strategy='uniform')
frac_lr,  mean_lr  = calibration_curve(y_binary, prob_lr,  n_bins=10, strategy='uniform')

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(mean_gnb, frac_gnb, 's-', color='steelblue', label='Naive Bayes')
ax.plot(mean_lr,  frac_lr,  's-', color='coral',     label='Logistic Regression')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title(f'Calibration Curve — "{class_names[0]}" vs Rest\n(Closer to diagonal = better calibrated)')
ax.legend()
plt.tight_layout()
plt.show()

print('If Naive Bayes clusters near 0 and 1: it is overconfident (independence assumption breaks).')
print('Logistic Regression should track the diagonal more closely.')

## 11. Feature Weight Visualisation (Logistic Regression)

Logistic Regression's weight matrix $\mathbf{W}$ (shape: `n_classes × n_features`) is directly interpretable.
Positive weight → feature pushes toward this class. Negative → pushes away.

In [ ]:
# ── Feature Weights per Class ──
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Logistic Regression Feature Weights per Class\n(Red = positive, Blue = negative)', fontsize=12)

for ax, cls in zip(axes.flat, range(len(class_names))):
    weights = lr.coef_[cls]
    colors = ['coral' if w > 0 else 'steelblue' for w in weights]
    ax.bar(range(len(weights)), weights, color=colors, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(class_names[cls], fontsize=9)
    ax.set_xlabel('Feature index', fontsize=7)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

# Top 5 most discriminative features
mean_abs = np.abs(lr.coef_).mean(axis=0)
top5 = np.argsort(mean_abs)[::-1][:5]
print('Top 5 most discriminative features (mean |weight| across all classes):')
for rank, idx in enumerate(top5):
    print(f'  {rank+1}. {feature_names[idx]:<32} | mean |weight| = {mean_abs[idx]:.4f}')

## 12. Regularisation Sweep — Effect of C on Accuracy

In [ ]:
# ── Regularisation Strength Experiment ──
C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
train_accs, test_accs, cv_means = [], [], []

for C in C_values:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(multi_class='multinomial', solver='lbfgs',
                                   C=C, max_iter=1000, random_state=SEED))
    ])
    pipe.fit(X_train_s, y_train)
    train_accs.append(accuracy_score(y_train, pipe.named_steps['lr'].predict(X_train_s)))
    test_accs.append(accuracy_score(y_test, pipe.named_steps['lr'].predict(X_test_s)))
    cv_means.append(cross_val_score(pipe, feature_matrix, labels, cv=cv, scoring='accuracy').mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(C_values, train_accs, 'o-', color='steelblue', label='Train Accuracy')
ax.semilogx(C_values, test_accs,  's-', color='coral',     label='Test Accuracy')
ax.semilogx(C_values, cv_means,   '^--', color='green',    label='CV Mean Accuracy')
ax.axhline(acc_gnb, color='gray', linestyle=':', label=f'Naive Bayes baseline ({acc_gnb:.3f})')
ax.set_xlabel('C (1/λ) — log scale')
ax.set_ylabel('Accuracy')
ax.set_title('Effect of Regularisation Strength C on Logistic Regression')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_C = C_values[np.argmax(test_accs)]
print(f'Best C on test set: {best_C}  (Accuracy: {max(test_accs):.4f})')
print('Use CV mean (not test accuracy) to pick C — avoids overfitting to the test set.')

## 13. Discussion Questions & Bonus Exercises

### 💬 Discussion Questions

**Q1.** Which classifier achieved higher accuracy on your dataset? What does this tell you about feature correlation in the HeLa features?

**Q2.** Look at the calibration curves. Which model is more overconfident? Why does the independence assumption in Naive Bayes cause poor probability calibration?

**Q3.** Are there specific organelle classes where Naive Bayes wins over Logistic Regression? What properties might those classes have?

**Q4.** Pick one class from the feature weight plot. Describe what the top positive and negative features say about that organelle's visual appearance.

**Q5.** In the regularisation sweep, what happens to train accuracy as C increases? Why does test accuracy eventually drop at very large C?

**Q6.** Both classifiers assume linearity. In what scenarios would SVM with an RBF kernel (Module 04) outperform both?

---

### 🚀 Bonus Exercises

**Bonus 1 — Small Data Regime**
Retrain with only 10 images per class. Does Naive Bayes hold up better? Why?

**Bonus 2 — Probability Calibration**
Use `CalibratedClassifierCV` to calibrate Naive Bayes probabilities. Does it improve the calibration curve?
```python
from sklearn.calibration import CalibratedClassifierCV
calibrated_gnb = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=5)
```

**Bonus 3 — Full Classifier Comparison**
Combine Module 04 and this module: build a bar chart comparing all 5 classifiers (Naive Bayes, Logistic Regression, Linear SVM, RBF SVM, k-NN).

**Bonus 4 — Feature Ablation Study**
Train Logistic Regression using only GLCM, only Geometric, only Haralick, then all combined. How much does each feature group contribute?

**Bonus 5 — Bernoulli Naive Bayes**
Try `BernoulliNB` after binarising features. Compare accuracy and calibration vs. GaussianNB.

In [ ]:
# ── Final Summary ──
print('=' * 55)
print('   TUTORIAL COMPLETE — KEY TAKEAWAYS')
print('=' * 55)
print()
print('1. Naive Bayes trains instantly via closed-form stats.')
print('   Works well with small data but assumes independence.')
print()
print('2. Logistic Regression trains via gradient descent.')
print('   Handles correlated features and gives better-calibrated')
print('   probabilities. Needs more data than Naive Bayes.')
print()
print('3. Always apply StandardScaler before Logistic Regression.')
print()
print('4. Evaluate with 5-fold CV, not just one train/test split.')
print()
print('5. Per-class accuracy + confusion matrix reveal more')
print('   than overall accuracy alone.')
print()
print('6. LR feature weights are directly interpretable —')
print('   use them to understand which features drive predictions.')
print('=' * 55)